In [ ]:
import os
import  json, random
import torch
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm
from sklearn.metrics import f1_score

from fine_tune.hebi_finetuner import HebiCellTypeClassifier
from fine_tune.supervised_loader import (
    get_slide_list,
    load_supervised_slide_dict,
    build_local_sampler,
    iterate_supervised_batches,
)


In [ ]:
import random
import numpy as np
import torch

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)


In [ ]:
foundation_ckpt = "./hebi_foundation_best3.pt"
data_root = "/DATA1/Roxana/Hierarchy/Two_levels/fine_tune/Liver_cosmax_normal"
output_dir = "./checkpoints"
os.makedirs(output_dir, exist_ok=True)

target_key = "y_niche"  # OR "y_niche"
device = torch.device("cuda:1" if torch.cuda.is_available() else "cpu")

batch_size = 64
num_workers = 4

n_epochs_linear =10
lr_head = 1e-3

do_finetune = True
n_epochs_finetune = 1
lr_backbone =1e-5

In [ ]:
lr_backbone = 1e-6

In [ ]:
niche_label_dict = {
    'Zone_1a/PV': 0,
    'Zone_1b': 1,
    'Zone_2a': 2,
    'Zone_2b': 3,
    'Zone_3a': 4,
    'Zone_3b/CV': 5,
    'interface': 6,
    'non-malignant': 7,
    'tumor': 8,
    'tumor subtype': 9
}
niche_label_healthy_dict = {
    'Zone_1a/PV': 0,
    'Zone_1b': 1,
    'Zone_2a': 2,
    'Zone_2b': 3,
    'Zone_3a': 4,
    'Zone_3b/CV': 5,
}
niche_label_cancer={'interface':0, 'non-malignant':1, 'tumor':2, 'tumor subtype':3}
region_label_dict = {
    'nan': 0
}
author_cell_type_dict = {
    'Antibody.secreting.B.cells': 0,
    'CD3+.alpha.beta.T.cells': 1,
    'Central.venous.LSECs': 2,
    'Cholangiocytes': 3,
    'Erthyroid.cells': 4,
    'Hep': 5,
    'Hep.1': 6, 
    'Hep.3': 7, 
    'Hep.4': 8, 
    'Hep.5': 9, 
    'Hep.6': 10,
    'Inflammatory.macrophages': 11, 
    'Mature.B.cells': 12, 
    'NK.like.cells': 13,
    'Non.inflammatory.macrophages': 14, 
    'NotDet': 15, 
    'Periportal.LSECs': 16,
    'Portal.endothelial.cells': 17, 
    'Stellate.cells': 18,
    'gamma.delta.T.cells.1': 19, 
    'tumor_1': 20, 
    'tumor_2': 21
}

if target_key == "y_class":
    n_classes = len(author_cell_type_dict)
else:
    n_classes = len(niche_label_healthy_dict)

print("Task:", target_key, "| n_classes:", n_classes)


In [ ]:
n_classes

In [ ]:
model = HebiCellTypeClassifier(
    n_classes=n_classes,
    dim_model=128,
    l0_encoder_type="transformer",
    global_num_layers=2,
    num_prototypes_lvl1=1024,
    num_prototypes_lvl2=20,
).to(device)

model.load_pretrained_weights(foundation_ckpt, device=device, freeze_backbone=True)

criterion = nn.CrossEntropyLoss()


In [ ]:
train_files = get_slide_list(data_root, split="train")
test_files  = get_slide_list(data_root, split="test")

print("Train slides:", len(train_files))
print("Test slides:", len(test_files))


In [ ]:
#11111111111111111
def run_one_epoch(files,device, optimizer=None, train=True, desc=""):
    if train:
        model.train()
    else:
        model.eval()

    all_true = []
    all_pred = []

    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    if train:
        random.shuffle(files)

    pbar = tqdm(files, desc=desc)

    for pt_path in pbar:
        try:
            slide = load_supervised_slide_dict(
                pt_path, target_key=target_key, map_location="cpu"
            )
        except Exception as e:
            print("Skipping:", pt_path, "Reason:", e)
            continue

        sampler = build_local_sampler(
            slide, batch_size=batch_size, num_workers=num_workers, shuffle=train
        )

        for (
            x_subgraph_raw,
            edge_index_subgraph,
            edge_attr_b,
            node2comm_batch,
            super_sp_global,
            gene_ids,
            y_seed,
            bs,
        ) in iterate_supervised_batches(slide, sampler, device=device, target_key=target_key):

            if train:
                optimizer.zero_grad(set_to_none=True)

            with torch.set_grad_enabled(train):
                logits = model(
                    x_subgraph_raw=x_subgraph_raw,
                    edge_index_subgraph=edge_index_subgraph,
                    edge_attr_b=edge_attr_b,
                    node2comm_batch=node2comm_batch,
                    super_sp_global=super_sp_global,
                    gene_ids=gene_ids,
                    bs=bs,
                    device=device
                )

                loss = criterion(logits, y_seed)

                if train:
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                    optimizer.step()

            total_loss += float(loss.item()) * bs
            preds = torch.argmax(logits, dim=1)

            all_true.append(y_seed.detach().cpu())
            all_pred.append(preds.detach().cpu())

            total_correct += int((preds == y_seed).sum().item())
            total_samples += int(bs)

        if total_samples > 0:
            pbar.set_postfix(
                loss=total_loss / total_samples,
                acc=total_correct / total_samples
            )

    y_true = torch.cat(all_true).numpy() if len(all_true) else np.array([])
    y_pred = torch.cat(all_pred).numpy() if len(all_pred) else np.array([])

    micro_f1 = f1_score(y_true, y_pred, average="micro") if y_true.size else 0.0
    macro_f1 = f1_score(y_true, y_pred, average="macro") if y_true.size else 0.0

    acc = (total_correct / total_samples) if total_samples > 0 else 0.0
    avg_loss = (total_loss / total_samples) if total_samples > 0 else 0.0

    return avg_loss, acc, micro_f1, macro_f1, y_true, y_pred


In [ ]:
#!!!!!!!!!!!!11111111111111111
print("\n❄️ PHASE 1: Linear probing (train + test only)")

optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=lr_head
)

for epoch in range(1, n_epochs_linear + 1):
    tr_loss, tr_acc, tr_micro, tr_macro, _, _ = run_one_epoch(
        train_files,device,
        optimizer=optimizer,
        train=True,
        desc=f"P1 Train {epoch}/{n_epochs_linear}"
    )

    te_loss, te_acc, te_micro, te_macro, y_true_l, y_pred_l = run_one_epoch(
        test_files,device,
        optimizer=None,
        train=False,
        desc=f"P1 Test  {epoch}/{n_epochs_linear}"
    )

    print(
        f"Epoch {epoch:02d} | "
        f"Train acc={tr_acc:.4f} macroF1={tr_macro:.4f} | "
        f"Test acc={te_acc:.4f} macroF1={te_macro:.4f}"
    )

final_linear_path = os.path.join(output_dir, "final_linear_model_class.pt")
#torch.save(model.state_dict(), final_linear_path)
print(f"💾 Saved final linear model → {final_linear_path}")


In [ ]:
################111111111111111
if do_finetune:
    print("\n🔥 PHASE 2: Fine-tuning")

    # Unfreeze everything
    for p in model.parameters():
        p.requires_grad = True

    # Fresh optimiser (Adam state reset), but weights continue from Phase 1
    optimizer = optim.Adam([
        {"params": model.gene_embedder.parameters(), "lr": lr_backbone},
        {"params": model.gene_linear.parameters(), "lr": lr_backbone},
        {"params": model.model_lvl0.parameters(), "lr": lr_backbone},
        {"params": model.model_lvl1.parameters(), "lr": lr_backbone},
        {"params": model.pooler_lvl2.parameters(), "lr": lr_backbone},
        {"params": model.model_lvl2.parameters(), "lr": lr_backbone},
        {"params": model.classifier_head.parameters(), "lr": lr_head},  # head learns faster
    ])

    # Train ONLY
    for epoch in range(1, n_epochs_finetune + 1):
        tr_loss, tr_acc, tr_micro, tr_macro, _, _ = run_one_epoch(
            train_files,device,
            optimizer=optimizer,
            train=True,
            desc=f"P2 Train {epoch}/{n_epochs_finetune}"
        )
        print(
            f"✅ P2 Ep {epoch} | "
            f"Train loss={tr_loss:.4f} | Train acc={tr_acc:.4f} | macroF1={tr_macro:.4f}"
        )

    # Test ONCE
    te_loss, te_acc, te_micro, te_macro, y_true, y_pred = run_one_epoch(
        test_files,device,
        optimizer=None,
        train=False,
        desc="P2 Final Test"
    )
    print(
        f"🎯 P2 Final Test | "
        f"Test loss={te_loss:.4f} | Test acc={te_acc:.4f} | macroF1={te_macro:.4f}"
    )

    finetuned_path = os.path.join(output_dir, "final_finetuned_model.pt")
    torch.save(model.state_dict(), finetuned_path)
    print(f"💾 Saved final finetuned model → {finetuned_path}")

else:
    print("\n⏹️ Skipping fine-tune.")



In [ ]:
finetuned_path = os.path.join(output_dir, "final_finetuned_model_niche.pt")
torch.save(model.state_dict(), finetuned_path)

In [ ]:
import json

with open("/DATA1/Roxana/Hierarchy/Two_levels/fine_tune/label_mappings.json") as f:
    mappings = json.load(f)

niche_to_id = mappings["niche_mapping"]
id_to_niche = {v: k for k, v in niche_to_id.items()}

niche_names = [id_to_niche[i] for i in range(len(id_to_niche))]


In [ ]:
niche_names

In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(
    y_true,
    y_pred,
    labels=niche_names
)

cm_norm = cm / cm.sum(axis=1, keepdims=True)


In [ ]:
y_pred

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, ax = plt.subplots(figsize=(14, 12))
im = ax.imshow(cm_norm, cmap="Blues", interpolation="nearest")
fig.colorbar(im, ax=ax)

ax.set_xticks(np.arange(len(niche_names)))
ax.set_yticks(np.arange(len(niche_names)))
ax.set_xticklabels(niche_names, rotation=45, ha="right", fontsize=7)
ax.set_yticklabels(niche_names, fontsize=7)

ax.set_xlabel("Predicted class")
ax.set_ylabel("True class")
ax.set_title("Normalised confusion matrix – class prediction (HEBI , test set)")

# numbers inside cells
for i in range(cm_norm.shape[0]):
    for j in range(cm_norm.shape[1]):
        val = cm_norm[i, j]
        if val > 0:
            ax.text(
                j, i, f"{val:.2f}",
                ha="center", va="center",
                color="white" if val > 0.5 else "black",
                fontsize=6
            )

plt.tight_layout()
plt.savefig(
    "confusion_matrix_class_HEBI_test.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

y_true = np.asarray(y_true, dtype=np.int64)
y_pred = np.asarray(y_pred, dtype=np.int64)

K = len(niche_names)

# matrix on IDs
cm = confusion_matrix(y_true, y_pred, labels=np.arange(K))
cm_norm = cm / (cm.sum(axis=1, keepdims=True) + 1e-12)

# plot
fig, ax = plt.subplots(figsize=(8, 5), dpi=600)
im = ax.imshow(cm_norm, aspect="auto")
plt.colorbar(im, ax=ax)

ax.set_title("Niche prediction confusion matrix")
ax.set_xlabel("Predicted")
ax.set_ylabel("True")

ax.set_xticks(np.arange(K))
ax.set_yticks(np.arange(K))
ax.set_xticklabels(niche_names, rotation=90, fontsize=6)
ax.set_yticklabels(niche_names, fontsize=6)

# =========================
# PRINT VALUES IN CELLS
# =========================
for i in range(K):
    for j in range(K):
        value = cm_norm[i, j]
        if value > 0.001:   # avoid printing tons of zeros
            ax.text(
                j, i,
                f"{value:.2f}",
                ha="center",
                va="center",
                fontsize=10,
                color= "black",
            )

plt.tight_layout()
plt.show()


In [ ]:
import scanpy as sc
import pandas as pd
cancer = sc.read_h5ad("/DATA1/Roxana/Data/cancer_liver_cosmax.h5ad")

In [ ]:
cancer = cosmx[cosmx.obs['Run_Tissue_name']=='CancerousLiver']

In [ ]:
import numpy as np
import pandas as pd

# your test FOVs
test_fovs = np.array([  2,   7,  38,  46,  53,  66,  69,  91, 125, 141, 156, 172, 177,
                      191, 201, 214, 222, 234, 239, 241, 251, 275, 279, 295, 302])

# ensure type matches your obs['fov'] (it's category of strings)
test_fov_set = set(map(str, test_fovs))

# boolean masks
is_test = healthy.obs["fov"].astype(str).isin(test_fov_set).to_numpy()
is_train = ~is_test

# store split labels
healthy.obs["split"] = pd.Categorical(np.where(is_test, "test", "train"),
                                      categories=["train", "test"])

print("Train cells:", int(is_train.sum()))
print("Test  cells:", int(is_test.sum()))
print("Unique test FOVs present:", healthy.obs.loc[is_test, "fov"].astype(str).nunique())

# (optional) create separate AnnData objects
healthy_train = healthy[is_train].copy()
healthy_test  = healthy[is_test].copy()


In [ ]:
cancer_test=cancer[cancer.obs["split"]=="test"]


In [ ]:
Fov=np.unique(cancer_test.obs.fov)

In [ ]:
d=cancer_test[cancer_test.obs['fov']=='266']


In [ ]:
y_pred_l

In [ ]:
cancer_test.obs['predicted_labels_f'] = y_pred

In [ ]:
cancer_test.obs['predicted_labels_f'] = cancer_test.obs['predicted_labels_f'].astype('category')


In [ ]:




import scanpy as sc
for f in Fov:
    d=cancer_test[cancer_test.obs['fov']==f]
    axs = sc.pl.spatial(
        d,
        spot_size=0.01,
        color="predicted_labels_f",
        title=f"Hebi (fine-tune)_{f}",
        show=False,
    )
    
    # axs can be a single Axes OR a list
    ax0 = axs[0] if isinstance(axs, list) else axs
    fig = ax0.figure
    fig.savefig(f"test_finetune/Hebi (fine-tune)_{f}_cancer.png", dpi=600, bbox_inches="tight")




In [ ]:

import scanpy as sc
for f in Fov:
    d=cancer_test[cancer_test.obs['fov']==f]
    axs = sc.pl.spatial(
        d,
        spot_size=0.01,
        color="niche_type",
        title=f"Annotated niche type {f}",
        show=False,
    )
    
    # axs can be a single Axes OR a list
    ax0 = axs[0] if isinstance(axs, list) else axs
    fig = ax0.figure
    fig.savefig(f"cancer_liver_annotation_test/annotated_niche_type_{f}_cancer.png", dpi=600, bbox_inches="tight")

